### Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!apt-get install unrar tree
!pip install ipecharts

### Data Extract

In [ ]:
!unrar x /content/drive/MyDrive/TUKL/data_online_line_width_alpha.rar /content/
!mv /content/data_online_line_width_alpha/ /content/hwd/

In [4]:
!tree -L 1 /content/hwd/

/content/hwd/
├── aux_cache
├── DATA_CARD.md
├── Dataset
├── main_1000.csv
├── main_1500.csv
├── main_2000.csv
├── main_2500.csv
├── main_3000.csv
├── main_3500.csv
├── main_5000.csv
├── main_6000.csv
├── test_cache
├── test_cache2
├── test_cache3
├── test.csv
├── test_extended.csv
├── test_leakproof.csv
├── test_leakproof_raw.csv
├── train_cache
├── train_cache2
├── train_cache3
├── train.csv
├── train_extended.csv
├── train_leakproof.csv
├── train_leakproof_raw.csv
├── val_cache
├── val_cache3
├── val.csv
├── val_extended.csv
├── val_leakproof.csv
└── val_leakproof_raw.csv

10 directories, 21 files


In [6]:
import pandas as pd

df = pd.read_csv('/content/hwd/main_1500.csv')
print(df.shape)
df.head(2)

(253, 7)


,id,cms,gender,age,csv,img,line
0,0,461136,f,21,Dataset/Data_1500/csv/csv_0000_0.csv,Dataset/Data_1500/img/img_0000_0.png,پڑھی گئیں کہ ٹرکوں کے ٹرک امدادی سامان سے بھرے
1,0,461136,f,21,Dataset/Data_1500/csv/csv_0000_1.csv,Dataset/Data_1500/img/img_0000_1.png,نہ دیتا تو آپؒ اس پر ناراضگی کا اظہار کرتے


In [10]:
data = pd.read_csv('/content/hwd/Dataset/Data_1500/csv/csv_0000_0.csv')
data.head()[['X cood.','Y cood.','Time']]

,X cood.,Y cood.,Time
0,15527,519,0
1,15468,518,2
2,15453,515,4
3,15440,504,6
4,15423,486,8


### Tensor Prep

In [12]:
import glob
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

# 1. Load and combine ALL main_*.csv files gracefully
all_csv_files = glob.glob('/content/hwd/main_*.csv')
df = pd.concat((pd.read_csv(f) for f in all_csv_files), ignore_index=True)

print(f"Loaded {len(all_csv_files)} files. Total dataset size: {len(df)} samples")

# 2. Dataset
class SimpleHWDataset(Dataset):
    def __init__(self, dataframe, base_dir='/content/hwd/'):
        self.df = dataframe.reset_index(drop=True)
        self.base_dir = base_dir
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Load points for this sample
        csv_path = self.base_dir + self.df.loc[idx, 'csv']
        points = pd.read_csv(csv_path)
        
        # Sort by Time to get the correct ground-truth sequence
        points = points.sort_values('Time')
        coords = points[['X cood.', 'Y cood.']].values.astype(np.float32)
        
        # Normalize between 0 and 1 (Required for models to learn spatial data easily)
        coords = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-6)
        
        # Shuffle inputs to simulate unordered OpenCV points
        shuffle_idx = np.random.permutation(len(coords))
        unordered_coords = coords[shuffle_idx]
        
        # Create target sequence (where each point actually belongs)
        target_seq = np.argsort(shuffle_idx)
        
        return torch.tensor(unordered_coords), torch.tensor(target_seq)

# 3. Simple Collate Function (for batch padding)
def simple_collate(batch):
    coords, targets = zip(*batch)
    
    # Pad sequences with 0s for coords and -1 for target labels (to ignore in loss)
    coords_padded = pad_sequence(coords, batch_first=True, padding_value=0.0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=-1)
    
    return coords_padded, targets_padded

# 4. Ready to use!
dataset = SimpleHWDataset(df)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=simple_collate)

# 4. Split the data
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 5. Create Datasets
train_dataset = SimpleHWDataset(train_df)
val_dataset = SimpleHWDataset(val_df)

# 6. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=simple_collate)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=simple_collate)

# Verify
x, y = next(iter(train_loader))
print("Batched Inputs shape (Batch, Seq_Len, X/Y):", x.shape)
print("Batched Targets shape (Batch, Seq_Len):", y.shape)

Loaded 8 files. Total dataset size: 2403 samples
Batched Inputs shape (Batch, Seq_Len, X/Y): torch.Size([16, 24592, 2])
Batched Targets shape (Batch, Seq_Len): torch.Size([16, 24592])
